In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [40]:

# ===============================
# Data loading and aggregation
# ===============================
def load_city_timeseries(data_dir: str) -> pd.DataFrame:
    """
    Load all building CSVs for a city and return total hourly load and generation.
    Assumes column3 = load, column30 = generation.
    """
    files = [f for f in os.listdir(data_dir) if f.endswith(".csv")]
    total_load, total_gen = None, None
    
    for i, fn in enumerate(files):
        df = pd.read_csv(os.path.join(data_dir, fn))
        if {"column3", "column30"}.issubset(df.columns):
            load = df["column3"].fillna(0).clip(lower=0)
            gen = df["column30"].fillna(0).clip(lower=0)
            if total_load is None:
                total_load, total_gen = load, gen
            else:
                total_load = total_load.add(load, fill_value=0)
                total_gen = total_gen.add(gen, fill_value=0)
        if (i+1) % 10 == 0:
            print(f"   🔄 Processed {i+1}/{len(files)} files in {data_dir}...")
    
    return pd.DataFrame({"load": total_load, "gen": total_gen})

# ===============================
# Offset calculations
# ===============================
def city_offset(ts: pd.DataFrame) -> pd.DataFrame:
    """Calculate baseline and cooperative offsets for a city (aggregated hourly)."""
    load, gen = ts["load"].values, ts["gen"].values
    deficit = np.maximum(load - gen, 0.0)
    surplus = np.maximum(gen - load, 0.0)
    selfuse = np.minimum(load, gen)
    return pd.DataFrame({
        "load": load, "gen": gen,
        "deficit": deficit, "surplus": surplus, "selfuse": selfuse
    })

def cross_city_offset(ts1: pd.DataFrame, ts2: pd.DataFrame) -> pd.DataFrame:
    """Combine two cities and compute cooperative offsets."""
    load = ts1["load"].values + ts2["load"].values
    gen = ts1["gen"].values + ts2["gen"].values
    deficit = np.maximum(load - gen, 0.0)
    surplus = np.maximum(gen - load, 0.0)
    selfuse = np.minimum(load, gen)
    return pd.DataFrame({
        "load": load, "gen": gen,
        "deficit": deficit, "surplus": surplus, "selfuse": selfuse
    })

# ===============================
# Save & Normalize
# ===============================
def normalize_series(series: pd.Series) -> pd.Series:
    """Normalize to [0,1] by dividing with its max."""
    max_val = series.max()
    return series / max_val if max_val > 0 else series

def run_analysis(city1, city1_path, city2, city2_path, out_csv="offset_results.csv"):
    """Main workflow: load, calculate, save CSV."""
    print(f"📂 Loading {city1}...")
    ts1 = load_city_timeseries(city1_path)
    print(f"📂 Loading {city2}...")
    ts2 = load_city_timeseries(city2_path)

    # City-level
    city1_res = city_offset(ts1)
    city2_res = city_offset(ts2)
    # Cross-city
    cross_res = cross_city_offset(ts1, ts2)

    # Merge into single dataframe
    result = pd.DataFrame({"hour": np.arange(len(ts1))})
    for col in city1_res.columns:
        result[f"{city1}_{col}"] = city1_res[col]
    for col in city2_res.columns:
        result[f"{city2}_{col}"] = city2_res[col]
    for col in cross_res.columns:
        result[f"{city1}+{city2}_{col}"] = cross_res[col]

    result.to_csv(out_csv, index=False)
    print(f"✅ Results saved to {out_csv}")
    return result


In [ ]:
if __name__ == "__main__":
    city1 = "KL"
    city2 = "PS"
    path1 = r".\Data\Simulation\KL_TS\CEAAgent"
    path2 = r".\Data\Simulation\PS_TS\CEAAgent"

    # Run analysis (CSV output)
    df_result = run_analysis(city1, path1, city2, path2, out_csv="offset_results.csv")



📂 Loading KL...
   🔄 Processed 10/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 20/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 30/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 40/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 50/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 60/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 70/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 80/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 90/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 100/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\K

In [59]:
import os
import numpy as np
import pandas as pd


# -----------------------------
# Load & aggregate one city (UTC-aware, skip bad files)
# -----------------------------
def load_city_timeseries(data_dir: str,
                         load_col: str = "column3",
                         gen_col: str = "column30",
                         log_bad_to: str | None = None) -> tuple[pd.DataFrame, list]:
    """
    Read all CSVs in `data_dir`, parse UTC time (e.g., '2024-01-01 09:00:00+00'),
    keep timezone-aware DatetimeIndex (UTC), hourly-resample to handle duplicates,
    and sum across buildings.

    Error handling:
      - If a file is missing 'time' OR has invalid 'time' values (any NaT after parsing),
        the file is skipped and its name is recorded in a bad-files list.

    Returns:
      totals_df: DataFrame indexed by UTC timestamps with columns ['load', 'gen'].
      bad_files: list of filenames that were skipped.

    If `log_bad_to` is provided, a CSV log of bad files is written there.
    """
    files = sorted([f for f in os.listdir(data_dir) if f.endswith(".csv")])
    if not files:
        raise FileNotFoundError(f"No CSV files found in: {data_dir}")

    totals = None
    bad_files: list[dict] = []

    for i, fn in enumerate(files):
        path = os.path.join(data_dir, fn)
        try:
            df = pd.read_csv(path)

            # 1) time column must exist
            if "time" not in df.columns:
                raise ValueError("missing 'time' column")

            # 2) parse time as tz-aware UTC
            ts = pd.to_datetime(df["time"], utc=True, errors="coerce")

            # 3) if any NaT present, treat this file as bad and skip
            if ts.isna().any():
                bad_examples = df.loc[ts.isna(), ["time"]].head(5).to_dict(orient="records")
                raise ValueError(f"invalid time values; examples={bad_examples}")

            # 4) build per-file series
            ser = pd.DataFrame(
                {
                    "load": pd.to_numeric(df.get(load_col, np.nan), errors="coerce"),
                    "gen":  pd.to_numeric(df.get(gen_col,  np.nan), errors="coerce"),
                },
                index=pd.DatetimeIndex(ts)  # tz-aware UTC
            )

            # 5) clean + aggregate duplicates within hour
            ser["load"] = ser["load"].clip(lower=0)
            ser["gen"]  = ser["gen"].clip(lower=0)
            ser = ser.resample("H").sum(min_count=1)

            # 6) merge into city totals
            totals = ser if totals is None else totals.add(ser, fill_value=0)

        except Exception as e:
            # Record bad file and continue
            bad_files.append({"file": fn, "reason": str(e)})
            continue

        if (i + 1) % 10 == 0 or i == len(files) - 1:
            print(f"   🔄 Processed {i+1}/{len(files)} files in {data_dir}...")

    # Write log if requested
    if log_bad_to is not None:
        pd.DataFrame(bad_files).to_csv(log_bad_to, index=False)

    if totals is None:
        # All files were bad
        raise RuntimeError(
            f"All files in {data_dir} failed time parsing. "
            f"See log: {log_bad_to}" if log_bad_to else "All files failed time parsing."
        )

    return totals.sort_index(), bad_files


# -----------------------------
# Metrics
# -----------------------------
def city_offset(ts: pd.DataFrame) -> pd.DataFrame:
    """Compute deficit, surplus, and self-use given ['load','gen']."""
    load = ts["load"].fillna(0)
    gen  = ts["gen"].fillna(0)
    out = pd.DataFrame(index=ts.index)
    out["load"]    = load
    out["gen"]     = gen
    out["deficit"] = (load - gen).clip(lower=0)
    out["surplus"] = (gen - load).clip(lower=0)
    out["selfuse"] = np.minimum(load, gen)
    return out


def cross_city_offset(ts1: pd.DataFrame, ts2: pd.DataFrame) -> pd.DataFrame:
    """Union-align two UTC time series and compute combined metrics."""
    idx = ts1.index.union(ts2.index)
    a = ts1.reindex(idx).fillna(0)
    b = ts2.reindex(idx).fillna(0)
    load = a["load"] + b["load"]
    gen  = a["gen"]  + b["gen"]
    return city_offset(pd.DataFrame({"load": load, "gen": gen}, index=idx))


# -----------------------------
# Quick QA (optional)
# -----------------------------
def qa_window(ts: pd.DataFrame, label: str, year: int, start="08-25", end="09-05"):
    """Quick stats around late-Aug to early-Sep to spot spikes."""
    t0 = pd.Timestamp(f"{year}-{start} 00:00", tz="UTC")
    t1 = pd.Timestamp(f"{year}-{end} 23:59:59", tz="UTC")
    win = ts.loc[(ts.index >= t0) & (ts.index <= t1)]
    if win.empty:
        print(f"[{label}] No data in window {t0}–{t1}.")
        return
    print(f"\n[{label}] {t0}–{t1}: "
          f"load max={win['load'].max():.3f}, gen max={win['gen'].max():.3f}, "
          f"deficit max={win.get('deficit', pd.Series([0])).max():.3f}")


# -----------------------------
# Main
# -----------------------------
def run_analysis(city1: str, city1_path: str,
                 city2: str, city2_path: str,
                 out_csv: str = "offset_results.csv",
                 load_col: str = "column3", gen_col: str = "column30") -> pd.DataFrame:
    print(f"📂 Loading {city1}...")
    ts1_raw, bad1 = load_city_timeseries(
        city1_path, load_col=load_col, gen_col=gen_col,
        log_bad_to=os.path.join(city1_path, "bad_files_log.csv")
    )
    city1_res = city_offset(ts1_raw)

    print(f"📂 Loading {city2}...")
    ts2_raw, bad2 = load_city_timeseries(
        city2_path, load_col=load_col, gen_col=gen_col,
        log_bad_to=os.path.join(city2_path, "bad_files_log.csv")
    )
    city2_res = city_offset(ts2_raw)

    cross_res = cross_city_offset(ts1_raw, ts2_raw)

    # Export on the union index (UTC)
    union_idx = cross_res.index
    c1 = city1_res.reindex(union_idx).add_prefix(f"{city1}_")
    c2 = city2_res.reindex(union_idx).add_prefix(f"{city2}_")
    cr = cross_res.add_prefix(f"{city1}+{city2}_")

    result = pd.DataFrame({"timestamp_utc": union_idx})
    for df_ in (c1, c2, cr):
        for col in df_.columns:
            result[col] = df_[col].values

    result.to_csv(out_csv, index=False)
    print(f"✅ Results saved to {out_csv}")

    # QA around Aug/Sep (uses first year present)
    base_year = pd.DatetimeIndex(union_idx).year.min()
    qa_window(city1_res, city1, base_year)
    qa_window(city2_res, city2, base_year)
    qa_window(cross_res, f"{city1}+{city2}", base_year)

    # ---- Bad-file summary ----
    total_bad = len(bad1) + len(bad2)
    print("\n====== Bad Files Summary ======")
    print(f"{city1}: {len(bad1)} bad file(s). Log -> {os.path.join(city1_path, 'bad_files_log.csv')}")
    print(f"{city2}: {len(bad2)} bad file(s). Log -> {os.path.join(city2_path, 'bad_files_log.csv')}")
    if total_bad > 0:
        print("Some examples:")
        for row in (bad1[:2] + bad2[:2]):  # show up to 4 examples
            print(f" - {row['file']}: {row['reason']}")
    print("================================")

    return result



In [ ]:

# -----------------------------
# Script entry point
# -----------------------------
if __name__ == "__main__":
    # Parameters
    city1 = "KL"
    city2 = "PS"
    path1 = r".\Data\Simulation\KL_TS\CEAAgent"
    path2 = r".\Data\Simulation\PS_TS\CEAAgent"

    # Run
    df_result = run_analysis(
        city1, path1,
        city2, path2,
        out_csv="offset_results_update.csv"
    )
    print(df_result.head())


📂 Loading KL...
   🔄 Processed 10/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 20/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 30/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 40/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 50/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 60/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 70/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 80/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 90/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\KL_TS\CEAAgent...
   🔄 Processed 100/7824 files in D:\c4e-jz713-a-tale-of-two-cities\Data\Simulation\K

In [ ]:
# Economics assumptions
YEARS = 25
PRICE_GROWTH = 0.0449
DISCOUNT_RATE = 0.0337
BUY_PRICE = 0.25     # €/kWh
FEEDIN_PRICE = 0.08  # €/kWh

def annual_cost_electric(deficit, surplus, buy_price, feedin_price):
    """Annual electricity cost: pay for deficit, subtract feed-in from surplus."""
    return deficit.sum() * buy_price - surplus.sum() * feedin_price

def npv_city_analysis(annual_saving: float, installed_area: float) -> tuple[float, list[float]]:
    """NPV calculation with your assumptions."""
    # CAPEX
    cost_base = 300 * installed_area + 800
    capex = cost_base + 0.25 * cost_base
    # OPEX
    opex = 2.5 * installed_area + 100

    cashflows = [-capex]
    npv = -capex
    for y in range(1, YEARS+1):
        benefit_y = annual_saving * ((1 + PRICE_GROWTH) ** y) - opex
        disc = benefit_y / ((1 + DISCOUNT_RATE) ** y)
        cashflows.append(disc)
        npv += disc
    return npv, cashflows

def payback_year(cashflows: list[float]) -> float | None:
    """Return first year when cumulative cashflow >= 0."""
    cum = 0.0
    for y, cf in enumerate(cashflows):
        cum += cf
        if cum >= 0:
            return float(y)
    return None

# ===============================
# Main
# ===============================

# 1. Load offset results
df = pd.read_csv(r"./offset_results.csv")

# 2. Compute annual baseline cost
cost_baseline = (
    annual_cost_electric(df["KL_deficit"], df["KL_surplus"], BUY_PRICE, FEEDIN_PRICE) +
    annual_cost_electric(df["PS_deficit"], df["PS_surplus"], BUY_PRICE, FEEDIN_PRICE)
)

# 3. Compute cooperative cost
cost_coop = annual_cost_electric(df["KL+PS_deficit"], df["KL+PS_surplus"], BUY_PRICE, FEEDIN_PRICE)

# 4. Annual saving
annual_saving = cost_baseline - cost_coop

# 5. Load roof areas from results files
kl_area_df = pd.read_csv(r".\Codes\results_kl.csv")
ps_area_df = pd.read_csv(r".\Codes\results_ps.csv")

installed_area_kl = kl_area_df.loc[kl_area_df["Property"] == "Roof solar suitable area", "Value"].sum()
installed_area_ps = ps_area_df.loc[ps_area_df["Property"] == "Roof solar suitable area", "Value"].sum()
installed_area = installed_area_kl + installed_area_ps

# 6. NPV
npv_val, cashflows = npv_city_analysis(annual_saving, installed_area)
payback = payback_year(cashflows)

# 7. Save summary
summary = pd.DataFrame([{
    "annual_cost_baseline": cost_baseline,
    "annual_cost_coop": cost_coop,
    "annual_saving": annual_saving,
    "installed_area": installed_area,
    "npv": npv_val,
    "payback_year": payback
}])

summary.to_csv("economic_summary.csv", index=False)
print("✅ Saved economic_summary.csv")


✅ Saved economic_summary.csv


In [ ]:
import os
from pathlib import Path
import pandas as pd

FOLDERS = [
    r".\Data\Simulation\KL_TS\CEAAgent",
    r".\Data\Simulation\PS_TS\CEAAgent",
]
OUTPUT_DIR = r".\Outputs"
TIME_COL = "time"
HEAT_COL = "column3" 
PV_COL   = "column30" 

YEAR = 2024
MIN_COVERAGE = 0.75  
# =====================================

def process_folder(folder: str) -> pd.DataFrame:
    folder = Path(folder)
   
    idx = pd.date_range(
        f"{YEAR}-01-01 00:00:00+00:00",
        f"{YEAR}-12-31 23:00:00+00:00",
        freq="H",
        tz="UTC",
    )
    agg = pd.DataFrame(index=idx, data={HEAT_COL: 0.0, PV_COL: 0.0})  

    
    for csv_path in sorted(folder.glob("*.csv")):
        try:
            df = pd.read_csv(csv_path, usecols=[TIME_COL, HEAT_COL, PV_COL])
        except Exception:
            
            continue

  
        ts = pd.to_datetime(df[TIME_COL], utc=True, errors="coerce")
        df = df.loc[~ts.isna(), [HEAT_COL, PV_COL]].copy()
        df.index = ts[~ts.isna()]
        if df.empty:
            continue


        df = df.apply(pd.to_numeric, errors="coerce")

   
        df = df.groupby(df.index.floor("H"))[[HEAT_COL, PV_COL]].sum(min_count=1)

     
        df = df.loc[df.index.year == YEAR]
        if df.empty:
            continue

        present_hours = df.index.nunique()
        coverage = present_hours / len(idx)
        if coverage < MIN_COVERAGE:
            continue  


        df = df.reindex(idx).fillna(0.0)


        agg[[HEAT_COL, PV_COL]] += df[[HEAT_COL, PV_COL]]
    agg = agg.rename(columns={HEAT_COL: "heat_kwh", PV_COL: "pv_kwh"})
    return agg

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    for folder in FOLDERS:
        agg = process_folder(folder)
        p = Path(folder)
        label = f"{p.parent.name}_{p.name}"
        out_path = Path(OUTPUT_DIR) / f"{label}_2024_hourly_aggregated.csv"

        out_df = agg.copy()
        out_df.insert(0, "time", out_df.index.astype(str))
        out_df.to_csv(out_path, index=False)
        print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()


Saved: D:\c4e-jz713-a-tale-of-two-cities\Outputs\KL_TS_CEAAgent_2024_hourly_aggregated.csv
Saved: D:\c4e-jz713-a-tale-of-two-cities\Outputs\PS_TS_CEAAgent_2024_hourly_aggregated.csv
